# Análise das notificações de dengue e chikungunya – exemplo

Notebook para explorar as fichas exportadas pelo formulário do tablet.
Os relatórios prontos ficam nos scripts (`scripts/relatorio_epidemiologico.py` e `scripts/boletim_publico.py`);
aqui é o lugar para perguntas novas.

**Como usar**

1. Copie os CSV exportados dos tablets para a pasta `entrada/`.
2. Rode as células de cima para baixo (Shift+Enter).
3. Se a pasta `entrada/` estiver vazia, o notebook usa as fichas fictícias de `dados/teste/`
   (gere com `python3 scripts/gerar_dados_teste.py 150`).

**Atenção (LGPD):** as tabelas têm dados pessoais. Antes de enviar o notebook para o GitHub,
limpe as saídas (menu *Edit → Clear Outputs of All Cells*).

## 1. Preparação

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Encontra a pasta do projeto, rodando o notebook de notebooks/ ou da raiz
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))

from comum import SINAIS, bairros_cnefe, ler_fichas, padronizar_bairro

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 4)

## 2. Leitura das fichas

In [ ]:
entrada = RAIZ / "entrada"
if not list(entrada.glob("*.csv")):
    entrada = RAIZ / "dados" / "teste"
    print("Pasta entrada/ vazia: usando fichas FICTÍCIAS de dados/teste/")

df, arquivos = ler_fichas([entrada])
df["bairro"] = padronizar_bairro(df, bairros_cnefe())
print(f"{len(df)} fichas lidas de {len(arquivos)} arquivo(s)")
df[["dt_notific", "dt_sin_pri", "se", "bairro", "classificacao", "faixa_etaria", "geo_metodo"]].head()

## 3. Visão geral

**Caso provável** = toda ficha, exceto as descartadas (classificação 5).

In [ ]:
df["classificacao"].value_counts().to_frame("fichas")

## 4. Curva epidêmica por semana de início dos sintomas

In [ ]:
curva = (df[df["provavel"]]
         .groupby(["ano_epi", "se"]).size()
         .rename("casos_provaveis").reset_index())
curva["semana"] = curva["ano_epi"].astype(int).astype(str) + "-SE" + curva["se"].astype(int).astype(str).str.zfill(2)

ax = curva.plot.bar(x="semana", y="casos_provaveis", legend=False, color="#d0621f")
ax.set_xlabel("Semana epidemiológica")
ax.set_ylabel("Casos prováveis")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
curva.tail()

## 5. Casos por bairro, com taxa por 1.000 domicílios (CNEFE 2022)

In [ ]:
dom = bairros_cnefe()[["bairro", "domicilios"]]
por_bairro = (df[df["provavel"]].groupby("bairro").size().rename("casos").reset_index()
              .merge(dom, on="bairro", how="left"))
por_bairro["por_1000_domicilios"] = (por_bairro["casos"] / por_bairro["domicilios"] * 1000).round(1)
por_bairro.sort_values("casos", ascending=False)

## 6. Tabela cruzada: classificação por faixa etária

In [ ]:
pd.crosstab(df["faixa_etaria"], df["classificacao"], margins=True, margins_name="Total")

## 7. Exemplo de pergunta nova

*Quantos casos prováveis tiveram febre **e** exantema, por bairro?*

Troque os sinais para fazer outras perguntas. Os códigos são `1 = Sim`, `2 = Não`.
Colunas disponíveis: veja a lista abaixo.

In [ ]:
print(", ".join(f"{c} ({n})" for c, n in SINAIS.items()))

filtro = df["provavel"] & (df["febre"] == "1") & (df["exantema"] == "1")
df[filtro].groupby("bairro").size().rename("casos").sort_values(ascending=False).to_frame()

## 8. Tempo entre o início dos sintomas e a notificação

In [ ]:
dias = df["dias_sint_notif"].dropna()
print(f"Mediana: {dias.median():.0f} dias | notificados em até 7 dias: {(dias <= 7).mean():.0%}")

ax = dias.plot.hist(bins=range(0, int(dias.max()) + 2), color="#0b6e4f", rwidth=.9)
ax.set_xlabel("Dias entre início dos sintomas e notificação")
ax.set_ylabel("Fichas")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 9. Mapa simples dos casos

Pontos cinza: endereços do IBGE (CNEFE 2022). Funciona sem internet.

In [ ]:
import csv
with open(RAIZ / "dados" / "cnefe_2022_4128302_uniflor.csv", encoding="utf-8") as f:
    fundo = pd.DataFrame([(float(r["LONGITUDE"]), float(r["LATITUDE"])) for r in csv.DictReader(f, delimiter=";")],
                         columns=["lon", "lat"])

pts = df[df["provavel"]].dropna(subset=["latitude", "longitude"])
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(fundo["lon"], fundo["lat"], s=3, color="#c9cfd4")
ax.scatter(pts["longitude"], pts["latitude"], s=20, color="#d0621f", edgecolor="#222", linewidth=.4)
# Área urbana (Centro); apague as duas linhas abaixo para ver o município inteiro
ax.set_xlim(-52.165, -52.151)
ax.set_ylim(-23.0925, -23.0795)
ax.set_aspect(1 / 0.92)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 10. Salvar um recorte

Os arquivos vão para `saida/`, que não é enviada ao GitHub.

In [ ]:
(RAIZ / "saida").mkdir(exist_ok=True)
recorte = df[filtro][["nm_pacient", "bairro", "nm_logrado", "nu_numero", "dt_sin_pri", "classificacao"]]
recorte.to_csv(RAIZ / "saida" / "recorte_febre_exantema.csv", sep=";", index=False, encoding="utf-8-sig")
print(f"{len(recorte)} linhas salvas em saida/recorte_febre_exantema.csv")